In [15]:
# We will run the notebook at root level.

import sys
from pathlib import Path

# notebook is one level below root
ROOT = Path.cwd().parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [3]:
# Utility functions for accessing and analyzing results

def load_paper_results(paper_id):
    """Load aggregated results for a specific paper."""
    results_file = RESULTS_FOLDER / paper_id / "aggregated" / "results.json"
    if results_file.exists():
        with open(results_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

def load_node_output(paper_id, node_name):
    """Load specific node output for a paper."""
    output_file = RESULTS_FOLDER / paper_id / "raw_outputs" / f"{node_name}.json"
    if output_file.exists():
        with open(output_file, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

def get_all_papers_gatekeeper_decisions():
    """Get gatekeeper decisions for all papers."""
    decisions = {}
    for paper_dir in RESULTS_FOLDER.glob("*/"):
        if (paper_dir / "aggregated" / "results.json").exists():
            results = load_paper_results(paper_dir.name)
            decisions[paper_dir.name] = results.get("gatekeeper_decision")
    return decisions

print("✅ Utility functions available:")
print("   - load_paper_results(paper_id): Load full results for a paper")
print("   - load_node_output(paper_id, node_name): Load specific node output")
print("   - get_all_papers_gatekeeper_decisions(): Get all gatekeeper decisions")

✅ Utility functions available:
   - load_paper_results(paper_id): Load full results for a paper
   - load_node_output(paper_id, node_name): Load specific node output
   - get_all_papers_gatekeeper_decisions(): Get all gatekeeper decisions


# Evaluation: Agent vs Ground Truth

In [4]:
import csv
import pandas as pd
from pathlib import Path

# Load ground truth benchmark

ROOT = Path(r"c:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow")
PILOT = ROOT / "Pilot_Evaluation"
BENCHMARK_FILE = PILOT / "DATA_sample_10/Data Science Research Process (DSRP) Framework.csv"

# Read CSV, treating first row as header
benchmark_df = pd.read_csv(BENCHMARK_FILE)

# Clean column names
benchmark_df.columns = benchmark_df.columns.str.strip()

print(f"✅ Loaded benchmark with {len(benchmark_df)} annotations")
print(f"\n📊 Benchmark columns ({len(benchmark_df.columns)}):")
for i, col in enumerate(benchmark_df.columns, 1):
    print(f"   {i}. {col}")

# Remove 'Any Comments (Optional)' columns
comment_cols = [col for col in benchmark_df.columns if 'Any Comments' in col or 'Any comments' in col]
benchmark_df_clean = benchmark_df.drop(columns=comment_cols)

print(f"\n🧹 Removed {len(comment_cols)} comment columns")
print(f"📊 Evaluation columns: {len(benchmark_df_clean.columns)}")

# Display sample
print("\n📋 Sample benchmark data:")
print(benchmark_df_clean[["Timestamp", "Paper ID", "Gatekeeper", "research_question"]].head())

✅ Loaded benchmark with 10 annotations

📊 Benchmark columns (45):
   1. Timestamp
   2. Evaluator Name
   3. Paper ID
   4. Gatekeeper
   5. Any Comments (Optional)
   6. research_question
   7. Comment (Optional)
   8. data_category
   9. data_format
   10. data_characteristics
   11. Any comments (Optional)
   12. Data Cleaning
   13. Data Reduction
   14. Data Transformation
   15. Any comments (Optional).1
   16. Foundational Paradigm
   17. Any Comments (Optional).1
   18. ML Learning Type
   19. ML Problem Type
   20. Deep Learning Used
   21. Any comments (optional)
   22. Specialised Paradigms (Skip if none)
   23. Any Comments (Optional).2
   24. evaluation_strategy
   25. learning_type
   26. problem_type
   27. evaluation_metrics_present
   28. validation_procedure
   29. effect_size_reported
   30. assumption_checks_reported
   31. Any comments (Optional).2
   32. explicit_theory
   33. implicit_theory_detected
   34. epistemological_orientation
   35. primary_research_orie

In [5]:
# Define mapping: Agent node output → Benchmark field
# Format: "agent_node": ("benchmark_column", "field_in_node_output", "field_type")
# field_type: "single", "multi_label", "binary"

AGENT_BENCHMARK_MAPPING = {
    "gatekeeper": {
        "benchmark_col": "Gatekeeper",
        "agent_field": "gatekeeper_decision",
        "field_type": "single"
    },
    "research_question": {
        "benchmark_col": "research_question",
        "agent_field": "dsrp_outputs.research_question.final_classification",  # Modify if different
        "field_type": "single"
    },
    "data_understanding": {
        "benchmark_col": "data_category",
        "agent_field": "dsrp_outputs.data_understanding.data_category",
        "field_type": "single"
    },
    "modelling": {
        "benchmark_col": "Foundational Paradigm",
        "agent_field": "dsrp_outputs.modelling.foundational_paradigm",
        "field_type": "single"
    },
    "evaluation_metrics_foundational_node": {
        "benchmark_col": "evaluation_metrics_present",
        "agent_field": "dsrp_outputs.evaluation_metrics.evaluation_metrics_present",
        "field_type": "single"
    },
    "evaluation_theoretical_orientation_node": {
        "benchmark_col": "epistemological_orientation",
        "agent_field": "dsrp_outputs.evaluation_theoretical_orientation.epistemological_orientation",
        "field_type": "single"
    },
    "evaluation_interpretability_node": {
        "benchmark_col": "interpretability_discussed",
        "agent_field": "interpretability_discussed",
        "field_type": "single"
    },
    "evaluation_ethical_social_node": {
        "benchmark_col": "bias_fairness_considered",
        "agent_field": "dsrp_outputs.evaluation_ethical_social.bias_fairness_considered",
        "field_type": "single"
    }
}

print("✅ Agent-Benchmark mapping defined")
print(f"📌 Mapped fields: {len(AGENT_BENCHMARK_MAPPING)}")
for agent_node, mapping in AGENT_BENCHMARK_MAPPING.items():
    print(f"   • {agent_node} → {mapping['benchmark_col']}")

✅ Agent-Benchmark mapping defined
📌 Mapped fields: 8
   • gatekeeper → Gatekeeper
   • research_question → research_question
   • data_understanding → data_category
   • modelling → Foundational Paradigm
   • evaluation_metrics_foundational_node → evaluation_metrics_present
   • evaluation_theoretical_orientation_node → epistemological_orientation
   • evaluation_interpretability_node → interpretability_discussed
   • evaluation_ethical_social_node → bias_fairness_considered


In [6]:
import json 

RESULTS_FOLDER = PILOT /"pilot_study_results"

In [7]:
def normalize_value(value):
    """Normalize values for comparison: handle empty, None, case sensitivity"""
    if pd.isna(value) or value == "" or str(value).strip() == "":
        return None
    return str(value).strip().lower()

def parse_multi_label(value):
    """Parse semicolon-separated values into set"""
    if pd.isna(value) or value == "":
        return set()
    values = str(value).split(";")
    return set(normalize_value(v) for v in values if normalize_value(v))

def extract_agent_values():
    """Extract agent predictions for all papers in results folder"""
    agent_data = {}
    
    for paper_dir in sorted(RESULTS_FOLDER.glob("*/")):
        if not paper_dir.is_dir():
            continue
        
        paper_id = paper_dir.name
        results_file = paper_dir / "aggregated" / "results.json"
        
        if not results_file.exists():
            continue
        
        with open(results_file, 'r', encoding='utf-8') as f:
            results = json.load(f)
        
        agent_data[paper_id] = results
    
    print(f"✅ Extracted agent outputs for {len(agent_data)} papers")
    return agent_data

# Extract agent data
agent_data = extract_agent_values()

# Create comparison dataframe
comparison_data = []

for _, row in benchmark_df_clean.iterrows():
    paper_id = normalize_value(row.get("Paper ID", ""))
    
    if not paper_id or paper_id not in agent_data:
        print(f"⚠️  Paper not found in agent results: {paper_id}")
        continue
    
    agent_results = agent_data[paper_id]
    
    comparison_row = {
        "Paper ID": paper_id,
        "Ground Truth": {},
        "Agent Output": {}
    }
    
    # Extract gatekeeper (binary)
    comparison_row["Ground Truth"]["gatekeeper"] = normalize_value(row.get("Gatekeeper", ""))
    comparison_row["Agent Output"]["gatekeeper"] = normalize_value(
        agent_results.get("gatekeeper_decision", "")
    )
    
    comparison_data.append(comparison_row)

comparison_df = pd.DataFrame(comparison_data)

print(f"\n📊 Created comparison dataset: {len(comparison_df)} papers")
print("\n🔍 Sample comparison:")
print(comparison_df[["Paper ID", "Ground Truth", "Agent Output"]].head())

✅ Extracted agent outputs for 10 papers

📊 Created comparison dataset: 10 papers

🔍 Sample comparison:
    Paper ID               Ground Truth               Agent Output
0   2011 - 1  {'gatekeeper': 'exclude'}  {'gatekeeper': 'exclude'}
1   2018 - 8  {'gatekeeper': 'include'}  {'gatekeeper': 'include'}
2  2018 - 12  {'gatekeeper': 'exclude'}  {'gatekeeper': 'include'}
3  2018 - 26  {'gatekeeper': 'exclude'}  {'gatekeeper': 'include'}
4  2019 - 33  {'gatekeeper': 'include'}  {'gatekeeper': 'include'}


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import numpy as np

def compute_metrics(y_true, y_pred, field_name, field_type="single"):
    """Compute classification metrics for a single field"""
    
    # Remove None pairs
    valid_pairs = [(t, p) for t, p in zip(y_true, y_pred) if t is not None and p is not None]
    
    if not valid_pairs:
        print(f"⚠️  No valid pairs for {field_name}")
        return None
    
    y_true_clean, y_pred_clean = zip(*valid_pairs)
    
    metrics = {
        "field": field_name,
        "field_type": field_type,
        "valid_samples": len(y_true_clean),
        "accuracy": accuracy_score(y_true_clean, y_pred_clean),
    }
    
    # Get unique classes
    classes = sorted(set(y_true_clean) | set(y_pred_clean))
    
    if len(classes) > 1:
        try:
            metrics["precision"] = precision_score(y_true_clean, y_pred_clean, average='weighted', zero_division=0)
            metrics["recall"] = recall_score(y_true_clean, y_pred_clean, average='weighted', zero_division=0)
            metrics["f1"] = f1_score(y_true_clean, y_pred_clean, average='weighted', zero_division=0)
        except:
            metrics["precision"] = metrics["recall"] = metrics["f1"] = None
    else:
        metrics["precision"] = metrics["recall"] = metrics["f1"] = 1.0
    
    return metrics

# Compute metrics for Gatekeeper
gatekeeper_true = [row["Ground Truth"]["gatekeeper"] for _, row in comparison_df.iterrows()]
gatekeeper_pred = [row["Agent Output"]["gatekeeper"] for _, row in comparison_df.iterrows()]

gatekeeper_metrics = compute_metrics(gatekeeper_true, gatekeeper_pred, "Gatekeeper", "binary")

print("=" * 80)
print("🎯 CLASSIFICATION METRICS: Gatekeeper (Binary)")
print("=" * 80)
if gatekeeper_metrics:
    for key, value in gatekeeper_metrics.items():
        if isinstance(value, float):
            print(f"{key:.<25} {value:.4f}")
        else:
            print(f"{key:.<25} {value}")

# Confusion matrix
if gatekeeper_metrics:
    from sklearn.metrics import ConfusionMatrixDisplay
    import matplotlib.pyplot as plt
    
    y_true_clean = [t for t, p in zip(gatekeeper_true, gatekeeper_pred) if t and p]
    y_pred_clean = [p for t, p in zip(gatekeeper_true, gatekeeper_pred) if t and p]
    
    cm = confusion_matrix(y_true_clean, y_pred_clean, labels=sorted(set(y_true_clean)))
    print(f"\n🔢 Confusion Matrix:")
    print(cm)

🎯 CLASSIFICATION METRICS: Gatekeeper (Binary)
field.................... Gatekeeper
field_type............... binary
valid_samples............ 10
accuracy................. 0.8000
precision................ 0.8444
recall................... 0.8000
f1....................... 0.7625

🔢 Confusion Matrix:
[[1 2]
 [0 7]]


In [9]:
# Comparison details
print("\n\n📋 DETAILED COMPARISON: Gatekeeper Decisions\n")
comparison_display = comparison_df.copy()
comparison_display["GT"] = comparison_display["Ground Truth"].apply(lambda x: x["gatekeeper"])
comparison_display["Agent"] = comparison_display["Agent Output"].apply(lambda x: x["gatekeeper"])
comparison_display["Match"] = comparison_display.apply(
    lambda row: "✅" if row["GT"] == row["Agent"] else "❌", axis=1
)

print(comparison_display[["Paper ID", "GT", "Agent", "Match"]].to_string(index=False))

# Summary statistics
matches = (comparison_display["GT"] == comparison_display["Agent"]).sum()
total = len(comparison_display)
print(f"\n\n📊 Overall Agreement: {matches}/{total} ({100*matches/total:.1f}%)")

# Per-class breakdown
print("\n📈 Per-Class Breakdown:")
for decision in sorted(set(comparison_display["GT"])):
    gt_count = (comparison_display["GT"] == decision).sum()
    correct = ((comparison_display["GT"] == decision) & (comparison_display["Agent"] == decision)).sum()
    accuracy = correct / gt_count if gt_count > 0 else 0
    print(f"   {decision:.<20} GT: {gt_count:2d} papers, Correct: {correct:2d}, Accuracy: {100*accuracy:.1f}%")



📋 DETAILED COMPARISON: Gatekeeper Decisions

 Paper ID      GT   Agent Match
 2011 - 1 exclude exclude     ✅
 2018 - 8 include include     ✅
2018 - 12 exclude include     ❌
2018 - 26 exclude include     ❌
2019 - 33 include include     ✅
 2020 - 8 include include     ✅
2021 - 28 include include     ✅
2021 - 56 include include     ✅
2023 - 58 include include     ✅
2024 - 99 include include     ✅


📊 Overall Agreement: 8/10 (80.0%)

📈 Per-Class Breakdown:
   exclude............. GT:  3 papers, Correct:  1, Accuracy: 33.3%
   include............. GT:  7 papers, Correct:  7, Accuracy: 100.0%


Go and find-out why the Agent classified above 2 paper incorrectly. Update the prompts and rule and check again whether now they are classified correctly. 

**Change Values Here like != to ==**

In [46]:
wrong_paper_ids = comparison_display[comparison_display["Match"] != "❌"]["Paper ID"].tolist()
print(f"\n⚠️  Papers with incorrect gatekeeper decisions: {len(wrong_paper_ids)}")
print("   " + ", ".join(wrong_paper_ids))


⚠️  Papers with incorrect gatekeeper decisions: 8
   2011 - 1, 2018 - 8, 2019 - 33, 2020 - 8, 2021 - 28, 2021 - 56, 2023 - 58, 2024 - 99


In [43]:
import os

# Change to ROOT directory so relative paths work
os.chdir(ROOT)
print(f"✅ Changed working directory to: {os.getcwd()}")

✅ Changed working directory to: C:\Users\sahil\OneDrive\PhD\3. Empirical Study\Methodological Workflow


In [47]:
from nodes.gatekeeper_node import gatekeeper_node
from utils.dsrp_state import DSRPState
import os

# Change to ROOT directory so relative paths in gatekeeper_node work
original_dir = os.getcwd()
os.chdir(ROOT)

print("🔄 RE-RUNNING GATEKEEPER ON WRONG PAPERS\n")

VECTOR_DB_PATH = PILOT / "pilot_study_10_vectors"
COLLECTION_NAME = "pilot_study_10"
EMBEDDING_MODEL = "text-embedding-3-small"

new_results = {}

for paper_id in wrong_paper_ids:
    print(f"🔄 Running gatekeeper on {paper_id}...")

    state = DSRPState(
        paper_id=paper_id,
        gatekeeper={}, # Initialize gatekeeper field
        dsrp_outputs={}, # Initialize dsrp_outputs field
        collection_name=COLLECTION_NAME,
        persist_directory=str(VECTOR_DB_PATH),
        embedding_model=EMBEDDING_MODEL
    )   
    
    try:
        result = gatekeeper_node(state)
        new_results[paper_id] = result["gatekeeper"]["final_classification"]
        print(f"  ✅ Result: {result['gatekeeper'].get("final_classification", 'N/A')}\n")
    except Exception as e:
        print(f"  ❌ Error: {e}\n")
        new_results[paper_id] = None

# Restore original directory
os.chdir(original_dir)

print(f"✅ Re-tested {len(new_results)} papers")

🔄 RE-RUNNING GATEKEEPER ON WRONG PAPERS

🔄 Running gatekeeper on 2011 - 1...
  ✅ Result: Exclude

🔄 Running gatekeeper on 2018 - 8...
  ✅ Result: Include

🔄 Running gatekeeper on 2019 - 33...
  ✅ Result: Include

🔄 Running gatekeeper on 2020 - 8...
  ✅ Result: Include

🔄 Running gatekeeper on 2021 - 28...
  ✅ Result: Include

🔄 Running gatekeeper on 2021 - 56...
  ✅ Result: Include

🔄 Running gatekeeper on 2023 - 58...
  ✅ Result: Include

🔄 Running gatekeeper on 2024 - 99...
  ✅ Result: Include

✅ Re-tested 8 papers


In [35]:
result["gatekeeper"]["final_classification"]

'Exclude'

**Notes:**
-	During evaluation first the first thing I noticed was the Agents were unable to correctly exclude some research papers Editorial (2018 – 12) and Conceptual papers (2018 – 26). While they both were Big Data and Information Technology, but they were not empirical papers which should have been excluded. Therefore, I refined the gatekeeper prompts, then re-run the analysis on those two papers. This time problem was sorted both papers were labelled as exclude by the agent. However, to verify if I have made screener too strict that even relevant papers get labelled as ‘exclude’. I re-ran the analysis (just the gatekeeper node) on remaining 8 papers which were labelled correctly before. While 7 studies were labelled correctly this time, 1 study (2024 – 99) was incorrectly labelled as exclude this time. 
